# 11 Registro rigido por mascaras - SB013

Objetivo: estimar una transformacion rigida inicial H&E -> HSI usando solo las mascaras binarias ya guardadas en `Imagenes_a_escala`.

En este baseline no se permite escala. La HSI es la imagen fija y la H&E es la imagen movil.

In [ ]:
from pathlib import Path
import csv
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

plt.rcParams['figure.dpi'] = 120

BASE_DIR = Path(r'Registration\DeepLearning')
INPUT_DIR = BASE_DIR / 'Imagenes_a_escala'
OUTPUT_DIR = BASE_DIR / 'Tecnicas_registration' / '00_baseline_clasico' / 'outputs' / 'outputs_registro_rigido_mascaras' / 'SB013'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_ID = 'SB013'
SUBJECT_DIR = INPUT_DIR / SUBJECT_ID

HE_RGB_PATH = SUBJECT_DIR / f'{SUBJECT_ID}_h&e.png'
HSI_RGB_PATH = SUBJECT_DIR / f'{SUBJECT_ID}_hsi.png'
HE_MASK_PATH = SUBJECT_DIR / f'{SUBJECT_ID}_he_mask.png'
HSI_MASK_PATH = SUBJECT_DIR / f'{SUBJECT_ID}_hsi_mask.png'
METADATA_PATH = SUBJECT_DIR / f'{SUBJECT_ID}_metadata.json'

print('Entrada:', SUBJECT_DIR)
print('Salida:', OUTPUT_DIR)

## Cargar RGB y mascaras

In [ ]:
def load_rgb(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)


def load_mask(path):
    return np.asarray(Image.open(path).convert('L')) > 0


he_rgb = load_rgb(HE_RGB_PATH)
hsi_rgb = load_rgb(HSI_RGB_PATH)
he_mask = load_mask(HE_MASK_PATH)
hsi_mask = load_mask(HSI_MASK_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))

print('H&E RGB:', he_rgb.shape, '| mask:', he_mask.shape, '| area:', int(he_mask.sum()))
print('HSI RGB:', hsi_rgb.shape, '| mask:', hsi_mask.shape, '| area:', int(hsi_mask.sum()))
print('px/cm comun:', metadata['target_px_per_cm'])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)
axes[0, 0].imshow(he_rgb)
axes[0, 0].set_title('H&E RGB movil')
axes[0, 1].imshow(hsi_rgb)
axes[0, 1].set_title('HSI RGB fija')
axes[1, 0].imshow(he_mask, cmap='gray')
axes[1, 0].set_title('Mascara H&E')
axes[1, 1].imshow(hsi_mask, cmap='gray')
axes[1, 1].set_title('Mascara HSI')
for ax in axes.ravel():
    ax.axis('off')
plt.show()

## Funciones de registro rigido

La transformacion rota la H&E alrededor de su centroide y despues traslada ese centroide al centroide de la HSI. Asi se explora solo la rotacion, sin escala.

In [ ]:
def centroid_xy(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError('Mascara vacia')
    return np.array([xs.mean(), ys.mean()], dtype=np.float64)


def rigid_matrix_moving_to_fixed(moving_mask, fixed_mask, angle_deg):
    moving_centroid = centroid_xy(moving_mask)
    fixed_centroid = centroid_xy(fixed_mask)
    matrix = cv2.getRotationMatrix2D(tuple(moving_centroid), float(angle_deg), 1.0)
    matrix[:, 2] += fixed_centroid - moving_centroid
    return matrix


def warp_to_fixed_canvas(image, fixed_shape, matrix, is_mask=False):
    interpolation = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    if is_mask:
        src = image.astype(np.uint8) * 255
        warped = cv2.warpAffine(
            src,
            matrix,
            dsize=(fixed_shape[1], fixed_shape[0]),
            flags=interpolation,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0,
        )
        return warped > 0

    return cv2.warpAffine(
        image,
        matrix,
        dsize=(fixed_shape[1], fixed_shape[0]),
        flags=interpolation,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )


def dice_score(mask_a, mask_b):
    a = np.asarray(mask_a, dtype=bool)
    b = np.asarray(mask_b, dtype=bool)
    denom = int(a.sum()) + int(b.sum())
    if denom == 0:
        return 1.0
    return float(2 * np.logical_and(a, b).sum() / denom)


def iou_score(mask_a, mask_b):
    a = np.asarray(mask_a, dtype=bool)
    b = np.asarray(mask_b, dtype=bool)
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(a, b).sum() / union)


def contour_mask(mask, thickness=1):
    mask_u8 = np.asarray(mask, dtype=np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = np.zeros_like(mask_u8)
    cv2.drawContours(out, contours, contourIdx=-1, color=255, thickness=int(thickness))
    return out > 0


def symmetric_contour_distance(mask_a, mask_b):
    ca = contour_mask(mask_a)
    cb = contour_mask(mask_b)
    if ca.sum() == 0 or cb.sum() == 0:
        return {'mean_px': np.nan, 'p95_px': np.nan}

    dist_to_b = cv2.distanceTransform((~cb).astype(np.uint8), cv2.DIST_L2, 5)
    dist_to_a = cv2.distanceTransform((~ca).astype(np.uint8), cv2.DIST_L2, 5)
    distances = np.concatenate([dist_to_b[ca], dist_to_a[cb]])
    return {
        'mean_px': float(np.mean(distances)),
        'p95_px': float(np.percentile(distances, 95)),
    }


def evaluate_angle(angle_deg):
    matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, angle_deg)
    warped_mask = warp_to_fixed_canvas(he_mask, hsi_mask.shape, matrix, is_mask=True)
    return {
        'angle_deg': float(angle_deg),
        'dice': dice_score(warped_mask, hsi_mask),
        'iou': iou_score(warped_mask, hsi_mask),
    }

## Busqueda angular

Primero se hace una busqueda gruesa cada 2 grados y despues una busqueda fina alrededor del mejor angulo.

In [ ]:
coarse_angles = np.arange(-180.0, 180.0 + 1e-6, 2.0)
coarse_results = [evaluate_angle(angle) for angle in coarse_angles]
best_coarse = max(coarse_results, key=lambda row: row['dice'])

fine_angles = np.arange(best_coarse['angle_deg'] - 3.0, best_coarse['angle_deg'] + 3.0 + 1e-6, 0.25)
fine_results = [evaluate_angle(angle) for angle in fine_angles]
best_fine = max(fine_results, key=lambda row: row['dice'])

angle_search_results = coarse_results + fine_results
print('Mejor angulo grueso:', best_coarse)
print('Mejor angulo fino:', best_fine)

In [ ]:
angles = np.array([row['angle_deg'] for row in angle_search_results])
dices = np.array([row['dice'] for row in angle_search_results])

order = np.argsort(angles)
plt.figure(figsize=(8, 3))
plt.plot(angles[order], dices[order], linewidth=1)
plt.axvline(best_fine['angle_deg'], color='red', linestyle='--', label=f"best {best_fine['angle_deg']:.2f} deg")
plt.xlabel('Angulo H&E -> HSI (grados)')
plt.ylabel('Dice mascara')
plt.title('Busqueda angular por solapamiento de mascaras')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Aplicar mejor transformacion y guardar resultados

In [ ]:
best_angle = best_fine['angle_deg']
best_matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, best_angle)

he_mask_rigid = warp_to_fixed_canvas(he_mask, hsi_mask.shape, best_matrix, is_mask=True)
he_rgb_rigid = warp_to_fixed_canvas(he_rgb, hsi_mask.shape, best_matrix, is_mask=False)

initial_matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, 0.0)
he_mask_initial = warp_to_fixed_canvas(he_mask, hsi_mask.shape, initial_matrix, is_mask=True)

contour_distance = symmetric_contour_distance(he_mask_rigid, hsi_mask)
metrics = {
    'subject_id': SUBJECT_ID,
    'registration_direction': 'he_to_hsi',
    'fixed_image': 'hsi',
    'moving_image': 'he',
    'best_angle_deg': float(best_angle),
    'transform_matrix_2x3': best_matrix.tolist(),
    'initial_centroid_aligned_dice_angle_0': dice_score(he_mask_initial, hsi_mask),
    'initial_centroid_aligned_iou_angle_0': iou_score(he_mask_initial, hsi_mask),
    'rigid_dice': dice_score(he_mask_rigid, hsi_mask),
    'rigid_iou': iou_score(he_mask_rigid, hsi_mask),
    'rigid_symmetric_contour_distance_mean_px': contour_distance['mean_px'],
    'rigid_symmetric_contour_distance_p95_px': contour_distance['p95_px'],
    'he_rgb_shape': list(he_rgb.shape),
    'hsi_rgb_shape': list(hsi_rgb.shape),
    'target_px_per_cm': float(metadata['target_px_per_cm']),
}

print(json.dumps(metrics, indent=2))

In [ ]:
def save_mask(path, mask):
    Image.fromarray((np.asarray(mask) > 0).astype(np.uint8) * 255).save(path)


def overlay_rgb_images(fixed_rgb, moving_rgb, fixed_mask=None, moving_mask=None):
    fixed_gray = cv2.cvtColor(fixed_rgb, cv2.COLOR_RGB2GRAY)
    moving_gray = cv2.cvtColor(moving_rgb, cv2.COLOR_RGB2GRAY)
    fixed_norm = cv2.normalize(fixed_gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    moving_norm = cv2.normalize(moving_gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    overlay = np.zeros((*fixed_gray.shape, 3), dtype=np.uint8)
    overlay[..., 0] = fixed_norm
    overlay[..., 1] = moving_norm
    overlay[..., 2] = fixed_norm // 3

    if fixed_mask is not None:
        overlay[contour_mask(fixed_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
    if moving_mask is not None:
        overlay[contour_mask(moving_mask, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)
    return overlay


overlay = overlay_rgb_images(hsi_rgb, he_rgb_rigid, hsi_mask, he_mask_rigid)
contours_overlay = np.zeros((*hsi_mask.shape, 3), dtype=np.uint8)
contours_overlay[contour_mask(hsi_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
contours_overlay[contour_mask(he_mask_rigid, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)

paths_out = {
    'he_rgb_rigid': OUTPUT_DIR / f'{SUBJECT_ID}_he_rigid_to_hsi.png',
    'he_mask_rigid': OUTPUT_DIR / f'{SUBJECT_ID}_he_mask_rigid_to_hsi.png',
    'overlay_rgb': OUTPUT_DIR / f'{SUBJECT_ID}_overlay_rgb_rigid.png',
    'overlay_contours': OUTPUT_DIR / f'{SUBJECT_ID}_overlay_contours_rigid.png',
    'metrics': OUTPUT_DIR / f'{SUBJECT_ID}_rigid_metrics.json',
    'angle_search': OUTPUT_DIR / f'{SUBJECT_ID}_angle_search.csv',
}

Image.fromarray(he_rgb_rigid).save(paths_out['he_rgb_rigid'])
save_mask(paths_out['he_mask_rigid'], he_mask_rigid)
Image.fromarray(overlay).save(paths_out['overlay_rgb'])
Image.fromarray(contours_overlay).save(paths_out['overlay_contours'])
paths_out['metrics'].write_text(json.dumps(metrics, indent=2), encoding='utf-8')

with paths_out['angle_search'].open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['angle_deg', 'dice', 'iou'])
    writer.writeheader()
    writer.writerows(angle_search_results)

for name, path in paths_out.items():
    print(name, '->', path)

## Visualizacion final

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
axes[0].imshow(hsi_rgb)
axes[0].set_title('HSI fija')
axes[1].imshow(he_rgb_rigid)
axes[1].set_title('H&E rigida al canvas HSI')
axes[2].imshow(overlay)
axes[2].set_title('Overlay final')
for ax in axes:
    ax.axis('off')
plt.show()

plt.figure(figsize=(5, 4))
plt.imshow(contours_overlay)
plt.title('Contornos: HSI rojo | H&E cyan')
plt.axis('off')
plt.show()

## Interpretacion

- Si el Dice mejora respecto a `angle_0`, la busqueda angular esta aportando informacion.
- Si el borde queda razonablemente orientado pero no encaja localmente, el siguiente paso es refinamiento afin o deformable.
- Si el mejor angulo parece visualmente incorrecto, conviene cambiar la metrica de busqueda: por ejemplo, usar mapas de distancia firmada o comparar contornos en vez de area completa.